In [1]:
import numpy as np
import json
import chipsplitting as cs
from chipsplitting import pairing_matrix, PascalForm, LinearForm
from chipsplitting.hyperfield import HyperfieldVector as HV, HyperfieldHomogeneousLinearSystem as HLinSystem, grid_iter, HyperfieldLinearForm

In [4]:
def countValidConfigsForContractions(pos_support_size, contraction_size, degree="even", use_extra_constraints = False):
    assert degree == "even" or degree == "odd", "degree must be 'even' or 'odd'"

    d = contraction_size * 3 - 1

    if d % 2 == 0 and degree == "odd":
        d += 1
    elif d % 2 == 1 and degree == "even":
        d += 1
        
    base_types = ["diag", "row", "col"]
    A = [PascalForm(d, b, k) for b in base_types for k in range(contraction_size)] + [PascalForm(d, b, k) for b in base_types for k in range(d - contraction_size + 1, d + 1)]
    
    if use_extra_constraints:
        # (1, d-1) see apple notes 3242444442
        A = A + [PascalForm(d, 'diag', i) - PascalForm(d, 'diag', j) for i,j in [(0,1), (0,2), (0,3), (0,4), (0,d-1), (0,d-2), (0,d-3), (0,d-4), (1,2), (1,3), (1, d), (1,d-1), (1,d-4), (1,d-2), (1,d-3), (2,d), (2,d-1), (2,d-3), (2, d-4), (3,d), (3,d-1), (3,d-2), (3, d-4)]]
        A = A + [PascalForm(d, 'diag', i) - PascalForm(d, 'diag', j) for i,j in [(d-4,d), (d-3,d), (d-2,d), (d-2,d-1), (d-1,d-2), (d-1,d-3), (d-1,d)]]
       
    A = [p.to_hyperfield().contract(contraction_size) for p in A if p != LinearForm.zero(d)]  
    linear_system = HLinSystem(A)
    solutions = linear_system.quick_solve_loop(pos_support_size)
    
    return solutions

def absolute(d, c):
    if c == 'd-1':
        return d-1
    if c == 'd-2':
        return d-2
    if c == 'd-3':
        return d-3
    if c == 'd-4':
        return d-4
    if c == 'd-0':
        return d
    return c

def rel(d, contraction_size, index):
    assert index < contraction_size or index > d - contraction_size
    return f"d-{d - index}" if index > contraction_size else index
    
def sign(x):
    return np.sign(x)

def has_inc_c(p, col, contraction_size):
    for row in range(contraction_size, p.degree - contraction_size - col):
        if p.support_pos[cs.utils.get_array_index(col, row)] > 0:
            if not p.support_pos[cs.utils.get_array_index(col, row)] < p.support_pos[cs.utils.get_array_index(col, row + 1)]:
                return False
        if p.support_neg[cs.utils.get_array_index(col, row)] > 0:
            if not p.support_neg[cs.utils.get_array_index(col, row)] > p.support_neg[cs.utils.get_array_index(col, row + 1)]:
                return False
    return True

def has_dec_c(p, col, contraction_size):
    for row in range(contraction_size, p.degree - contraction_size - col):
        if p.support_pos[cs.utils.get_array_index(col, row)] > 0:
            if not p.support_pos[cs.utils.get_array_index(col, row)] > p.support_pos[cs.utils.get_array_index(col, row + 1)]:
                return False
        if p.support_neg[cs.utils.get_array_index(col, row)] > 0:
            if not p.support_neg[cs.utils.get_array_index(col, row)] < p.support_neg[cs.utils.get_array_index(col, row + 1)]:
                return False
    return True

def is_constant(p, col, contraction_size):
    for row in range(contraction_size, p.degree - contraction_size - col):
        if p.support_pos[cs.utils.get_array_index(col, row)] != p.support_pos[cs.utils.get_array_index(col, contraction_size)]:
            return False

        if p.support_neg[cs.utils.get_array_index(col, row)] != p.support_neg[cs.utils.get_array_index(col, contraction_size)]:
            return False
    return True

def is_contractable(p, contraction_size = 5):
    assert p.degree >= contraction_size * 3 - 1

    # check b
    for row in range(contraction_size):
        for col in range(contraction_size, p.degree - contraction_size - row + 1):
            if sign(p.support_pos[cs.utils.get_array_index(col, row)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size, row)]):
                return False

            if sign(p.support_neg[cs.utils.get_array_index(col, row)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size, row)]):
                return False

    # check c
    for col in range(contraction_size):
        for row in range(contraction_size, p.degree - contraction_size - col + 1):
            if sign(p.support_pos[cs.utils.get_array_index(col, row)]) != sign(p.support_pos[cs.utils.get_array_index(col, contraction_size)]):
                return False

            if sign(p.support_neg[cs.utils.get_array_index(col, row)]) != sign(p.support_neg[cs.utils.get_array_index(col, contraction_size)]):
                return False

    # check d1
    for j in range(contraction_size):
        for i in range(contraction_size, p.degree - j - contraction_size + 1, 2):
            if sign(p.support_pos[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size, p.degree - j - contraction_size)]):
                return False
                
            if sign(p.support_neg[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size, p.degree - j - contraction_size)]):
                return False

    # check d2
    for j in range(contraction_size):
        for i in range(contraction_size + 1, p.degree - j - contraction_size + 1, 2):
            if sign(p.support_pos[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size + 1, p.degree - j - contraction_size - 1)]):
                return False
                
            if sign(p.support_neg[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size + 1, p.degree - j - contraction_size - 1)]):
                return False

    return True


"""
    Given an expression of pascal forms, find all units such that the expression is contractable.
"""
def find_contractables(lin_combinations, ops, contraction_size):
    import itertools
    
    d_begin = contraction_size * 3 - 1 + 3
    d_end = contraction_size * 3 - 1 + 6
        
    res = []
    combinations = {}

    for d in range(d_begin, d_end + 1):
        units = list(range(contraction_size)) + list(range(d - contraction_size + 1, d + 1))
        bases = [[(PascalForm(d, b, k), rel(d, contraction_size, k)) for k in units] for b in lin_combinations]

        for comb in itertools.product(*bases):
            form = LinearForm.zero(d)
            units = []
            for i, t in enumerate(comb):
                p, u = t
                units.append(u)
                if i == 0 or ops[i-1] == "+":
                    form += p
                else:
                    form -= p
            units = tuple(units)
            if units not in combinations:
                combinations[units] = {}
            combinations[units][d] = {"contractable": is_contractable(form), "form": form.to_hyperfield().contract(contraction_size)}
                
    for units, dict in combinations.items():
        is_valid = True
        for d in dict.keys():
            if d % 2 == 0:
                if dict[d]["contractable"] == False or dict[d]["form"] != dict[d_begin if d_begin % 2 == 0 else d_begin + 1]["form"]:
                    is_valid = False
                    break 
            else:
                if dict[d]["contractable"] == False or dict[d]["form"] != dict[d_begin if d_begin % 2 == 1 else d_begin + 1]["form"]:
                    is_valid = False
                    break
                
        if is_valid:
            res.append(units)

    return res

def hyperfield_vector_from_support(d, support_pos, support_neg):
    w = [-1] + [0] * (d-1)
    for x in support_pos:
        w[x] = 1
    return HV(w)
    
def is_root(hyperfield_forms, w):
    for p in hyperfield_forms:
        y = p(w)
        if not np.isnan(y) and y != 0:
            return False
    return True



def invert(p):
    return HyperfieldLinearForm(p.support_neg, p.support_pos)

In [3]:
%%time
n = 6
contraction_size = 5
res1 = countValidConfigsForContractions(n, contraction_size, "even", use_extra_constraints = True)
print(f"Number of configurations for even d: {len(res1)}")

res2 = countValidConfigsForContractions(n, contraction_size, "odd", use_extra_constraints = True)
print(f"Number of configurations for odd d: {len(res2)}")

Number of configurations for even d: 106806
Number of configurations for odd d: 110272
CPU times: user 2.16 s, sys: 15.2 ms, total: 2.18 s
Wall time: 2.18 s


In [97]:
def is_form_col_constant(form, col):
    mode, unit = form
    if mode == "col":
        return False
    return unit == col

def is_form_row_constant(form, row):
    mode, unit = form
    if mode == "row":
        return False
    return unit == row
    
def is_expression_col_constant(modes, units, ops, col):
    # only row and diag are relevant
    relevant = [(m,u,op) for m,u,op in zip(modes, units, ["+"] + list(ops)) if (m == "row" or m == "diag") and type(u) is int]
    if col == 0 and (tuple(relevant) == (("diag",1,"-"), ("row",1,"+")) or tuple(relevant) == (("diag",1,"+"), ("row",1,"-"))):
        return True
        
    for mode, unit, op in relevant:
        if col < unit:
            return False
    return True

def is_expression_col_zero(modes, units, ops, col):
    # only row and diag are relevant
    relevant = [(m,u,op) for m,u,op in zip(modes, units, ["+"] + list(ops)) if (m == "row" or m == "diag") and type(u) is int]
    if not relevant:
        return True
        
    if col ==1 and (relevant == [("diag",1,"-"), ("row",1,"+")] or relevant == [("diag",1,"+"), ("row",1,"-")] ):
        return True
    
    for mode, unit, op in relevant:
        if col <= unit:
            return False
    
    return True

In [90]:
%%time
d = 16
contraction_size = 5

combinations = [
    (("row", "row"), ("-")),
    (("row", "col"), ("-")),
    (("row", "diag"), ("-")),
    (("col", "col"), ("-")),
    (("col", "diag"), ("-")),
    (("diag", "diag"), ("-")),
    (("diag", "diag", "col", "row"), ("-", "+", "+")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "+")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "-")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "-", "-")),
    #(("diag", "diag", "col", "col", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        if (units[0] == units[1] and comb[0] == comb[1] and ops[0] == "-") or (ops[-1] == "-" and len(ops) == 4 and units[2] == units[4] and comb[2] == comb[4]):
            continue
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            # check c
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if is_constant(p,col,5):
                        if not is_expression_col_constant(comb, units, ops, col):
                            print(f"col constant Proof did not work, {comb}, {units}, {col}")
                    #elif has_inc_c(p, col, 5):
                    #    print("inc:", units)
                    #elif has_dec_c(p, col, 5):
                    #    print("dec:", units)
                    #else:
                    #    print(p)
                    #    assert False
                else:
                    # zero column
                    if not is_expression_col_zero(comb, units, ops, col):
                        print(f"zero Proof did not work, {comb}, {units}, {col}")
print("Success")

Success
CPU times: user 38.3 s, sys: 384 ms, total: 38.7 s
Wall time: 38.4 s


In [91]:
%%time
d = 16
contraction_size = 5

combinations = [
    (("diag", "diag", "col", "row", "col"), ("-", "+", "+", "+")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "-")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "-", "-")),
    #(("diag", "diag", "col", "col", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        if units[0] == units[1] and comb[0] == comb[1] and ops[0] == "-":
            continue
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            # check c
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if is_constant(p,col,5):
                        if not is_expression_col_constant(comb, units, ops, col):
                            print(f"col constant Proof did not work, {comb}, {units}, {col}")
                    #elif has_inc_c(p, col, 5):
                    #    print("inc:", units)
                    #elif has_dec_c(p, col, 5):
                    #    print("dec:", units)
                    #else:
                    #    print(p)
                    #    assert False
                else:
                    # zero column
                    if not is_expression_col_zero(comb, units, ops, col):
                        print(f"zero Proof did not work, {comb}, {units}, {col}")
print("Success")

Success
CPU times: user 6min 30s, sys: 7.11 s, total: 6min 37s
Wall time: 6min 31s


In [93]:
%%time
d = 16
contraction_size = 5

combinations = [
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "+")),
    (("diag", "diag", "col", "row", "col"), ("-", "+", "+", "-")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "-", "-")),
    #(("diag", "diag", "col", "col", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        if units[0] == units[1] or units[2] == units[4]:
            # skip diag(i) - diag(i) and col(i) - col(i)
            continue
        if units[1] == 0 and units[3] == 0:
            # skip -diag(0) + row(0)
            continue
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            # check c
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if is_constant(p,col,5):
                        if not is_expression_col_constant(comb, units, ops, col):
                            print(f"col constant Proof did not work, {comb}, {units}, {col}")
                    #elif has_inc_c(p, col, 5):
                    #    print("inc:", units)
                    #elif has_dec_c(p, col, 5):
                    #    print("dec:", units)
                    #else:
                    #    print(p)
                    #    assert False
                else:
                    # zero column
                    if not is_expression_col_zero(comb, units, ops, col):
                        print(f"zero Proof did not work, {comb}, {units}, {col}")
print("Success")

Success
CPU times: user 6min 34s, sys: 6.02 s, total: 6min 40s
Wall time: 6min 37s


In [98]:
%%time
d = 16
contraction_size = 5

combinations = [
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "+")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "-")),
    (("diag", "diag", "col", "row", "col"), ("-", "+", "-", "-")),
    #(("diag", "diag", "col", "col", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        if units[0] == units[1] or units[2] == units[4]:
            # skip diag(i) - diag(i) and col(i) - col(i)
            continue
        if units[1] == 0 and units[3] == 0:
            # skip -diag(0) + row(0)
            continue
        if units[0] == 0 and units[3] == 0:
            # skip diag(0) - row(0)
            continue
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            # check c
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if is_constant(p,col,5):
                        if not is_expression_col_constant(comb, units, ops, col):
                            print(f"col constant Proof did not work, {comb}, {units}, {col}")
                    #elif has_inc_c(p, col, 5):
                    #    print("inc:", units)
                    #elif has_dec_c(p, col, 5):
                    #    print("dec:", units)
                    #else:
                    #    print(p)
                    #    assert False
                else:
                    # zero column
                    if not is_expression_col_zero(comb, units, ops, col):
                        print(f"zero Proof did not work, {comb}, {units}, {col}")
print("Success")

Success
CPU times: user 6min 37s, sys: 5.12 s, total: 6min 42s
Wall time: 6min 40s


In [99]:
%%time
d = 16
contraction_size = 5

combinations = [
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "+")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "-")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "-", "-")),
    (("diag", "diag", "col", "col", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        if units[0] == units[1] or units[2] == units[4]:
            # skip diag(i) - diag(i) and col(i) - col(i)
            continue
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            # check c
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if is_constant(p,col,5):
                        if not is_expression_col_constant(comb, units, ops, col):
                            print(f"col constant Proof did not work, {comb}, {units}, {col}")
                    #elif has_inc_c(p, col, 5):
                    #    print("inc:", units)
                    #elif has_dec_c(p, col, 5):
                    #    print("dec:", units)
                    #else:
                    #    print(p)
                    #    assert False
                else:
                    # zero column
                    if not is_expression_col_zero(comb, units, ops, col):
                        print(f"zero Proof did not work, {comb}, {units}, {col}")
print("Success")

Success
CPU times: user 6min 48s, sys: 6.59 s, total: 6min 55s
Wall time: 6min 51s
